# 9. Create an adjusted measure and save outputs

Raw counts answer:

> How many libraries fall inside each census area?

But census areas contain very different numbers of residents.

We can also calculate libraries per 10,000 residents.

In [ ]:
import geopandas as gpd

In [ ]:
libraries = gpd.read_file(
    "../data/processed/toronto_libraries_clean.gpkg"
)

census = gpd.read_file(
    "../data/raw/toronto_census_areas.geojson"
)

libraries = libraries.to_crs(census.crs)
points = libraries.copy()
points["geometry"] = points.geometry.representative_point()

joined = gpd.sjoin(
    points,
    census[["ADAUID", "geometry"]],
    how="left",
    predicate="within"
)

counts = (
    joined.groupby("ADAUID")
    .size()
    .reset_index(name="library_count")
)

result = census.merge(
    counts,
    on="ADAUID",
    how="left"
)

result["library_count"] = result["library_count"].fillna(0)

In [ ]:
result["libraries_per_10000"] = (
    result["library_count"]
    / result["pop_2021_count"]
    * 10000
)

This rate is still only a rough measure.

It does **not** account for:

- library size or capacity;
- travel distance;
- transit access;
- barriers created by roads or geography;
- differences in population need;
- people using libraries outside their home census area.

In [ ]:
result.to_file(
    "../data/processed/census_library_analysis.gpkg",
    driver="GPKG"
)

result[
    [
        "ADAUID",
        "pop_2021_count",
        "hh_no_veh_pct",
        "library_count",
        "libraries_per_10000"
    ]
].to_csv(
    "../output/census_library_summary.csv",
    index=False
)